# Intra-Cell Assignment using CQF

This notebook demonstrates quantum-based resource block (RB) allocation to users under a single access point.
- Real-valued throughput is encoded as phase
- RB exclusivity is enforced by oracles
- The most efficient RB-UE assignment is found via quantum amplitude amplification

In [ ]:
import numpy as np
from qiskit import Aer, transpile, assemble
from qiskit.visualization import plot_histogram
from qiskit.visualization import circuit_drawer
from qiskit.providers.aer import AerSimulator

from qiskit_oracle.oracle import build_rb_exclusivity, build_phase_oracle
from qiskit_oracle.driver import encode_state, run_amplitude_amplification, decode_counts

import matplotlib.pyplot as plt
import os

os.makedirs("fig", exist_ok=True)


## Step 1: Define the problem (UEs, RBs, and throughput matrix)

In [ ]:
num_ue = 3
num_rb = 6

# Throughput matrix: UE rows × RB columns
throughput_matrix = np.array([
    [85,  93, 120, 118, 101, 90],
    [77,  88,  95,  94,  87, 89],
    [80,  85, 102, 103, 92, 100]
])
r_min, r_max = throughput_matrix.min(), throughput_matrix.max()


## Step 2: Encode and build the quantum circuit

In [ ]:
# Build state register
state, aux, qc = encode_state(num_ue, num_rb)

# Apply exclusivity and throughput encoding
qc = build_rb_exclusivity(qc, state, aux)
qc = build_phase_oracle(qc, state, throughput_matrix, r_min, r_max)

# Save full circuit
circuit_drawer(qc, output='mpl').savefig("fig/intra_circuit.png")


## Step 3: Run amplitude amplification

In [ ]:
qc_amplified = run_amplitude_amplification(qc, num_iter=3)

# Simulate
backend = AerSimulator()
qc_amplified.save_statevector()
job = backend.run(qc_amplified, shots=8192)
result = job.result()
counts = result.get_counts()

# Save histogram
plot_histogram(counts).savefig("fig/intra_histogram.png")

# Decode result
best = decode_counts(counts, num_ue, num_rb)[0]
print("Best UE-RB assignment:", best)
